In [ ]:
from neural_net.model_config import save_model_config, get_config, ModelConfig
from neural_net.trainers import SimpleTrainer, KFoldTrainer
import tensorflow as tf

In [8]:
from argparse import Namespace
from pathlib import Path
args = Namespace(
    workdir=Path("/eos/user/a/anunezde/Z_OUTPUT_eos/Run3_0626/Vars_EvenEvs"),
    rostername="test_fewEvs",
    outdirname="playground",
    trainer="simple",
    pass_idx=None,
    config_name="multi_HH_ttbar_tW_3j",
    log_level="info",
)

In [11]:
model_config = get_config(args.config_name, args.rostername)
outdir = args.workdir / args.outdirname
modeldir = outdir / model_config.name
modeldir.mkdir(exist_ok=True, parents=True)
save_model_config(model_config, modeldir / 'config.yml')

TRAINERS = {'simple': SimpleTrainer, 'kfold': KFoldTrainer}
TrainerClass = TRAINERS.get(args.trainer, None)
if TrainerClass is None:
    raise ValueError(f"Unknown trainer: {args.trainer}")

print(f"Tensorflow version: {tf.__version__}")

trainer = TrainerClass(model_config, args.workdir, modeldir)

Available top-level keys: ['SL_3j_resolved_vars', 'SL_4j_resolved_vars', 'models']
Loading models from test_fewEvs.yml...
	multi_HH_ttbar_tW_3j
	multi_HH_ttbar_tW_4j
Tensorflow version: 2.13.1

Model Config: multi_HH_ttbar_tW_3j
Directory: /eos/user/a/anunezde/Z_OUTPUT_eos/Run3_0626/Vars_EvenEvs/playground/multi_HH_ttbar_tW_3j


In [24]:
train_data, val_data, test_data = trainer.get_data()
train_data = train_data.prefetch(tf.data.AUTOTUNE)
val_data = val_data.prefetch(tf.data.AUTOTUNE)


Dataset metadata after enriching:
                                     File            Tree                    Process  Total Events  Process Event Ratio  Take events  DS Events  DS Ratio     GenWeight  Class  Class Index  Class DS Ratio   SampleWeight
0          ggHH_kl_0_kt_1_hh_bbww_dl_2022  SL_3j_resolved     ggHH_kl_0_kt_1_hh_bbww          2366             0.052731      1000000       2302  0.012191  1.603617e+02     HH            0        0.013634    2498.991943
1        ggHH_kl_0_kt_1_hh_bbww_dl_2022EE  SL_3j_resolved     ggHH_kl_0_kt_1_hh_bbww          8180             0.182308      1000000       7937  0.042033  5.527728e+02     HH            0        0.047008    8614.120117
2          ggHH_kl_0_kt_1_hh_bbww_dl_2023  SL_3j_resolved     ggHH_kl_0_kt_1_hh_bbww          7345             0.163699      1000000       7120  0.037706  4.959855e+02     HH            0        0.042169    7729.175781
3      ggHH_kl_0_kt_1_hh_bbww_dl_2023BPix  SL_3j_resolved     ggHH_kl_0_kt_1_hh_bbww     

2025-07-16 05:15:18.185711: W tensorflow/core/framework/op_kernel.cc:1816] UNKNOWN: KeyError: 3
Traceback (most recent call last):

  File "/cvmfs/sft.cern.ch/lcg/views/LCG_105/x86_64-el9-gcc11-opt/lib/python3.9/site-packages/tensorflow/python/ops/script_ops.py", line 268, in __call__
    ret = func(*args)

  File "/cvmfs/sft.cern.ch/lcg/views/LCG_105/x86_64-el9-gcc11-opt/lib/python3.9/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "/cvmfs/sft.cern.ch/lcg/views/LCG_105/x86_64-el9-gcc11-opt/lib/python3.9/site-packages/tensorflow/python/data/ops/from_generator_op.py", line 290, in finalize_py_func
    generator_state.iterator_completed(iterator_id)

  File "/cvmfs/sft.cern.ch/lcg/views/LCG_105/x86_64-el9-gcc11-opt/lib/python3.9/site-packages/tensorflow/python/data/ops/dataset_ops.py", line 876, in iterator_completed
    del self._iterators[self._normalize_id(iterator_id)]

KeyError: 3


2025-07-16 05:15:18.186282: W t


Training dataset:
   Class    count percentage Sample Weight
0     HH  101,307    89.417%      113634.3
1  ttbar    5,995     5.291%      990948.6
2     tW    5,995     5.291%      529679.3
3  Total  113,297                         

Validation dataset:
   Class   count percentage Sample Weight
0     HH  33,769    89.419%       38078.8
1  ttbar   1,998     5.291%      245977.9
2     tW   1,998     5.291%      139683.8
3  Total  37,765                         

Test dataset:
   Class   count percentage Sample Weight
0     HH  33,769    89.412%       37118.0
1  ttbar   2,000     5.295%      273713.8
2     tW   1,999     5.293%       85956.2
3  Total  37,768                         


In [25]:
from neural_net import utils as nn_utils
from neural_net.model_design import ModelNetwork, ModelEvaluator
train_mean, train_var, train_samples = nn_utils.log_training_stats(train_data, trainer.config, trainer.logger)
network = ModelNetwork(trainer.config, trainer.traindir, trainer.logger, train_samples)
model = network.build_model(train_mean, train_var)


The ignore_value is -9999


2025-07-16 05:16:11.327129: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 111617 of 11000000
2025-07-16 05:16:11.507436: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.



Training data statistics (total samples: 113297):
            Feature      mean    variance
0            met_pt   63.4759   1975.3670
1           met_phi    0.0036      3.3122
2         nAK4_btag    1.4675      0.2925
3       ak4_jet0_pt  116.7706   3594.7264
4      ak4_jet0_phi   -0.0059      3.3078
5      ak4_jet0_eta   -0.0034      1.2061
6   ak4_jet0_bscore    0.6274      0.1808
7       ak4_jet1_pt   67.0935   1132.8239
8      ak4_jet1_phi   -0.0012      3.2958
9      ak4_jet1_eta    0.0017      1.2919
10  ak4_jet1_bscore    0.3822      0.1871
11      ak4_jet2_pt   41.0687    292.5161
12     ak4_jet2_phi    0.0043      3.2973
13     ak4_jet2_eta    0.0024      1.3807
14  ak4_jet2_bscore    0.2895      0.1543
15          lep0_pt   60.9026   1158.9034
16         lep0_phi   -0.0030      3.2894
17         lep0_eta   -0.0061      1.1916
18         lep0_iso    0.0092      0.0011
19          blnu_mT  235.6725   8174.2681
20          blnu_pt  118.2233   3493.5842
21     blnu_bl_mInv  117.

In [26]:
# trained_model, history = network.fit(model, train_data, val_data)
train_data_pruned = train_data.map(lambda d: (d["features"], d["class_oh"], d["sample_weight"]))
val_data_pruned = val_data.map(lambda d: (d["features"], d["class_oh"], d["sample_weight"]))
steps_per_epoch = network.train_samples // network.batch_size
network.logger.info(f"Steps per epoch: {steps_per_epoch}")

Steps per epoch: 110


In [ ]:
from neural_net.model_design import LoggingCallbackNew, LoggingCallback
def get_callbacks(outdir, logger, using_validation: bool = False) -> list[tf.keras.callbacks.Callback]:
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_plateau = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5)
    terminate_on_nan = tf.keras.callbacks.TerminateOnNaN()

    last_ckpt_file = str(outdir / "last_checkpoint.keras")
    print(last_ckpt_file)
    last_ckpt_cb = tf.keras.callbacks.ModelCheckpoint(
        filepath= last_ckpt_file,
        save_weights_only=False,
        save_best_only=False,
        save_freq='epoch'   # Save last after every epoch
    )

    best_ckpt_file = str(outdir / "best_checkpoint.keras")
    print(best_ckpt_file)
    best_ckpt_cb = tf.keras.callbacks.ModelCheckpoint(
        filepath= best_ckpt_file,
        monitor='val_loss',
        mode='min',
        save_weights_only=False,
        save_best_only=True  # Save only the best one
    )

    callbacks = [
        LoggingCallbackNew(logger), terminate_on_nan, 
        last_ckpt_cb, 
        # best_ckpt_cb
    ]
    if using_validation:
        callbacks.extend([early_stopping, reduce_plateau])
    return callbacks

In [29]:
history = model.fit(
    x=train_data_pruned,
    epochs=network.epochs,
    callbacks = callbacks,
    validation_data=val_data_pruned,
    verbose=2
)

Epoch 1/3


2025-07-16 05:18:32.779999: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 111617 of 11000000
2025-07-16 05:18:33.022958: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.


Epoch 1   - loss: 14.7033, accuracy:  0.2001, precision:  0.1934, recall:  0.1449, auc_roc:  0.3385, val_loss:  9.4285, val_accuracy:  0.1697, val_precision:  0.1671, val_recall:  0.1188, val_auc_roc:  0.2628


ValueError: The following argument(s) are not supported with the native Keras format: ['options']

In [ ]:
network.package_best_model()
network.plot_training_curves(history)